In [ ]:
#Clear and move WB data to silver
from pyspark.sql.types import LongType, IntegerType
from pyspark.sql import functions as F

df = spark.read.table("LakehouseBronze.Bronze.WB")
df_wb = df[["date","value","country_id","country_value","indicator_value"]]
df_wb = df_wb.withColumn("value", F.col("value").cast(LongType())).withColumnRenamed("value", "Income")
df_wb = df_wb.withColumn("date", F.col("date").cast(IntegerType())).withColumnRenamed("date", "Year")
df_wb = df_wb.withColumnsRenamed({"country_id": "CountryID", "country_value": "CountryValue", "indicator_value": "Indicator"})
df_wb = df_wb.dropna().dropDuplicates()
df_wb.write.format("delta").mode("overwrite").saveAsTable("LakehouseSilver.Silver.WB")

In [ ]:
#Clear and move Currency data to silver
from pyspark.sql.functions import col
from pyspark.sql.types import DecimalType
from pyspark.sql import functions as F

df = spark.read.table("LakehouseBronze.Bronze.FX")
df_curr = df[["TIME_PERIOD", "OBS_VALUE"]]
df_curr = df_curr.withColumn("OBS_VALUE", col("OBS_VALUE").cast(DecimalType(10,4))).withColumnRenamed("OBS_VALUE", "CurrencyValue")
df_curr = df_curr.withColumn("TIME_PERIOD", F.date_format("TIME_PERIOD", "yyyy/MM/dd")).withColumnRenamed("TIME_PERIOD", "Date")
df_curr = df_curr.withColumn("CurrencyConvert", F.lit("USD-EUR"))
df_curr = df_curr.dropna().dropDuplicates()
df_curr.write.format("delta").mode("overwrite").saveAsTable("LakehouseSilver.Silver.FX")

In [ ]:
#Clear and move OpenAQ data to silver
from pyspark.sql.types import DecimalType
from pyspark.sql import functions as F

sensor_names = ["O3", "PM25"]

for i in sensor_names:
    df = spark.read.table(f"LakehouseBronze.Bronze.OpenAQ_{i}")
    df = df.dropna(subset=("value", "period_datetimeTo_utc")).dropDuplicates()
    df = df.withColumnsRenamed({"value": "MeasureValue", "period_datetimeTo_utc": "MeasureDateTime", "name": "MeasureName", "units": "Unit", "sensor_id": "SensorID"})
    df = df.withColumns({"MeasureValue": F.col("MeasureValue").cast(DecimalType(6,3)), "MeasureDateTime": F.date_format(F.col("MeasureDateTime"), "yyyy/MM/dd hh:mm:ss")})
    df = df.withColumn("MeasureLocation", F.when(F.col("SensorID") == 1128, "Brooklyn").when(F.col("SensorID").isin([1152, 1106]), "Queens").when(F.col("SensorID").isin([673, 671]), "Manhattan").when(F.col("SensorID") == 1146, "Bronx").otherwise("Unknown"))
    df = df[["MeasureValue", "MeasureDateTime", "MeasureName", "Unit", "SensorID", "MeasureLocation"]]
    df.write.format("delta").mode("append").saveAsTable("LakehouseSilver.Silver.OpenAQ")

In [ ]:
#Clear, unite and move TLC data to silver - FHV
import notebookutils
from time import time 
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType
from pyspark.errors import PySparkException

months = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
fhv_list = [f"fhv_{year}_{month}" for year in range(2015, 2027) for month in months]

for i in fhv_list:
    try:
        if not spark.catalog.tableExists(f"LakehouseBronze.Bronze_TLC.{i}"):
            continue
        df = spark.read.table("LakehouseBronze.Bronze_TLC.fhv_2015_01")
        df = df.dropna(subset=("pickup_datetime", "dropOff_datetime")).dropDuplicates().withColumn("dropOff_datetime", F.when(F.col("dropOff_datetime") < F.col("pickup_datetime"), F.col("pickup_datetime")).otherwise(F.col("dropOff_datetime")))
        df = df.fillna(0,("PULocationID", "DOLocationID")).withColumns(
                {"PULocationID": F.col("PULocationID").cast(IntegerType()), "DOLocationID": F.col("DOLocationID").cast(IntegerType())}
                ).withColumnsRenamed({"PULocationID": "PickLocation", "DOLocationID": "DropLocation"})
        df = df.withColumnsRenamed({"pickup_datetime": "PickDate", "dropOff_datetime": "DropDate"})
        df = df.withColumn("PickDate", F.date_format(F.col("PickDate"), "yyyy/MM/dd hh:mm:ss"))
        df = df.withColumn("DropDate", F.date_format(F.col("DropDate"), "yyyy/MM/dd hh:mm:ss"))
        df = df[["PickLocation", "DropLocation", "PickDate", "DropDate"]]
        df = df.withColumns({"TLCType": F.lit("FHV"), "Fare": F.lit(0).cast(DecimalType(8,2)), "Distance": F.lit(0).cast(DecimalType(8,3)), "Passengers": F.lit(0).cast(IntegerType())})
        df.write.format("delta").mode("append").saveAsTable("LakehouseSilver.Silver.TLC")
    except (Exception, PySparkException) as e:
        notebookutils.fs.append("Files/tlc_log.txt", e, True)
        time.sleep(0.5)

In [ ]:
#Clear, unite and move TLC data to silver - FHVHV
import notebookutils
from time import time 
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType
from pyspark.errors import PySparkException


months = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
fhvhv_list = [f"fhvhv_{year}_{month}" for year in range(2019, 2027) for month in months]

for i in fhvhv_list:
    try:
        if not spark.catalog.tableExists(f"LakehouseBronze.Bronze_TLC.{i}"):
            continue
        df = spark.read.table(f"LakehouseBronze.Bronze_TLC.{i}")
        df = df.dropna(subset=("pickup_datetime", "dropoff_datetime")).dropDuplicates().withColumn("dropoff_datetime", F.when(F.col("dropoff_datetime") < F.col("pickup_datetime"), F.col("pickup_datetime")).otherwise(F.col("dropoff_datetime")))
        df = df.fillna(0,("PULocationID", "DOLocationID")).withColumns(
            {"PULocationID": F.col("PULocationID").cast(IntegerType()), "DOLocationID": F.col("DOLocationID").cast(IntegerType())}
            ).withColumnsRenamed({"PULocationID": "PickLocation", "DOLocationID": "DropLocation"})
        if "cbd_congestion_fee" not in df.columns:
            df = df.withColumn("cbd_congestion_fee", F.lit(0))
        df = df.fillna(0, ("base_passenger_fare", "tolls", "bcf", "sales_tax", "congestion_surcharge", "airport_fee", "tips", "cbd_congestion_fee"))
        df = df.withColumnsRenamed({"pickup_datetime": "PickDate", "dropoff_datetime": "DropDate", "trip_miles": "Distance"})
        df = df.withColumn("Fare", F.round(F.col("base_passenger_fare") + F.col("tolls") + F.col("bcf") + F.col("sales_tax") + F.col("congestion_surcharge") + F.col("airport_fee") + F.col("tips") + F.col("cbd_congestion_fee"),2).cast(DecimalType(8,2)))
        df = df.withColumn("PickDate", F.date_format(F.col("PickDate"), "yyyy/MM/dd hh:mm:ss"))
        df = df.withColumn("DropDate", F.date_format(F.col("DropDate"), "yyyy/MM/dd hh:mm:ss"))
        df = df[["PickLocation", "DropLocation", "PickDate", "DropDate","Distance", "Fare"]]
        df = df.withColumns({"Passengers": F.lit(0).cast(IntegerType()), "TLCType": F.lit("FHVHV"), "Distance": F.col("Distance").cast(DecimalType(8,3))})
        df.write.format("delta").mode("append").saveAsTable("LakehouseSilver.Silver.TLC")
        print(f"{i} Done!")
    except (Exception, PySparkException) as e:
        notebookutils.fs.append("Files/tlc_log.txt", e, True)
        time.sleep(0.5)

In [ ]:
#Clear, unite and move TLC data to silver - green
import notebookutils
from time import time 
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType
from pyspark.errors import PySparkException

months = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
green_list = [f"green_{year}_{month}" for year in range(2014, 2027) for month in months]

for i in green_list:
    try:
        if not spark.catalog.tableExists(f"LakehouseBronze.Bronze_TLC.{i}"):
            continue
        df = spark.read.table(f"LakehouseBronze.Bronze_TLC.{i}")
        df = df.dropna(subset=("lpep_pickup_datetime", "lpep_dropoff_datetime")).dropDuplicates().withColumn("lpep_dropoff_datetime", F.when(F.col("lpep_dropoff_datetime") < F.col("lpep_pickup_datetime"), F.col("lpep_pickup_datetime")).otherwise(F.col("lpep_dropoff_datetime")))
        df = df.fillna(0,("PULocationID", "DOLocationID", "total_amount")).withColumns(
                {"PULocationID": F.col("PULocationID").cast(IntegerType()), "DOLocationID": F.col("DOLocationID").cast(IntegerType())}
                ).withColumnsRenamed({"PULocationID": "PickLocation", "DOLocationID": "DropLocation"})
        df = df.withColumnsRenamed({"lpep_pickup_datetime": "PickDate", "lpep_dropoff_datetime": "DropDate", "trip_distance": "Distance", "total_amount": "Fare", "passenger_count": "Passengers"})
        df = df.withColumns({"TLCType": F.lit("Green"), "Distance": F.col("Distance").cast(DecimalType(8,3)), "Fare": F.col("Fare").cast(DecimalType(8,2)), "Passengers": F.col("Passengers").cast(IntegerType())})
        df = df.withColumn("PickDate", F.date_format(F.col("PickDate"), "yyyy/MM/dd hh:mm:ss"))
        df = df.withColumn("DropDate", F.date_format(F.col("DropDate"), "yyyy/MM/dd hh:mm:ss"))
        df = df[["PickLocation", "DropLocation", "PickDate", "DropDate", "Distance", "Fare", "Passengers", "TLCType"]]
        df.write.format("delta").mode("append").saveAsTable("LakehouseSilver.Silver.TLC")
        print(f"{i} Done!")
    except (Exception, PySparkException) as e:
        notebookutils.fs.append("Files/tlc_log.txt", e, True)
        time.sleep(0.5)

In [ ]:
#Clear, unite and move TLC data to silver - yellow
import notebookutils
from time import time 
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType
from pyspark.errors import PySparkException

months = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
yellow_list = [f"yellow_{year}_{month}" for year in range(2011, 2027) for month in months]
#Skipped 2009, 2010 with different attributes

for i in yellow_list:
    try:
        if not spark.catalog.tableExists(f"LakehouseBronze.Bronze_TLC.{i}"):
            continue
        df = spark.read.table(f"LakehouseBronze.Bronze_TLC.{i}")
        df = df.dropna(subset=("tpep_pickup_datetime", "tpep_dropoff_datetime")).dropDuplicates().withColumn("tpep_dropoff_datetime", F.when(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime"), F.col("tpep_pickup_datetime")).otherwise(F.col("tpep_dropoff_datetime")))
        df = df.fillna(0,("PULocationID", "DOLocationID", "total_amount")).withColumns(
                {"PULocationID": F.col("PULocationID").cast(IntegerType()), "DOLocationID": F.col("DOLocationID").cast(IntegerType())}
                ).withColumnsRenamed({"PULocationID": "PickLocation", "DOLocationID": "DropLocation"})
        df = df.dropna(subset="total_amount")
        df = df.withColumnsRenamed({"tpep_pickup_datetime": "PickDate", "tpep_dropoff_datetime": "DropDate", "trip_distance": "Distance", "total_amount": "Fare", "passenger_count": "Passengers"})
        df = df.withColumns({"TLCType": F.lit("Yellow"), "Distance": F.col("Distance").cast(DecimalType(8,3)), "Fare": F.col("Fare").cast(DecimalType(8,2)), "Passengers": F.col("Passengers").cast(IntegerType())})
        df = df.withColumn("PickDate", F.date_format(F.col("PickDate"), "yyyy/MM/dd hh:mm:ss"))
        df = df.withColumn("DropDate", F.date_format(F.col("DropDate"), "yyyy/MM/dd hh:mm:ss"))
        df = df[["PickLocation", "DropLocation", "PickDate", "DropDate", "Distance", "Fare", "Passengers","TLCType"]]
        df.write.format("delta").mode("append").saveAsTable("LakehouseSilver.Silver.TLC")
        print(f"{i} Done!")
    except (Exception, PySparkException) as e:
        notebookutils.fs.append("Files/tlc_log.txt", e, True)
        time.sleep(0.5)

In [ ]:
from pyspark.sql import functions as F

df = spark.read.table("LakehouseBronze.Bronze.Zones")
df = df.withColumn("IsAirport", F.when(F.col("LocationID").isin([1, 132, 138]), 1).otherwise(0))
df = df.withColumnsRenamed({"Zone": "ZoneName", "Borough": "Neighborhood", "service_zone": "ServiceZone"})
df.write.format("delta").mode("overwrite").saveAsTable("LakehouseSilver.Silver.Zones")